# 01 DataCheck

## 0. Research Questions

**Research question:** Is the proportion of current cigarette use different between students who
felt sad or hopeless and those who did not?

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_cycle3_q8_cleaned.csv"
TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

raw = pd.read_csv(RAW_PATH)

print("Rows, columns:", raw.shape)

Rows, columns: (14041, 103)


## 2. Variable Definitions

### 2.1 Group variable: SadOrHopeless

- **Variable name:** `SadOrHopeless`
- **What the variable measures:** whether the student felt so sad or hopeless almost every day for 2 weeks or more in a row during the past 12 months that they stopped doing some usual activities.
- **Valid codes used:**  
  - `1` = Yes  
  - `2` = No
- **Recoding rule used in this project:**  
  - `1` -> `sad_binary = 1`  
  - `2` -> `sad_binary = 0`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.
- **Final valid sample size for proportion  analysis:** 13845

In [2]:
sad_raw = raw["SadOrHopeless"]
current_cig_raw = raw["CurrentCigaretteUse"]

# ------------------------------------------------------------
# Recode explanatory/group variable: SadOrHopeless
# 1 = Yes, felt sad or hopeless
# 2 = No, did not feel sad or hopeless
# ------------------------------------------------------------
raw["sad_binary"] = pd.NA

raw.loc[sad_raw.eq(1), "sad_binary"] = 1
raw.loc[sad_raw.eq(2), "sad_binary"] = 0

raw["sad_group"] = raw["sad_binary"].map({
    1: "Sad/Hopeless: Yes",
    0: "Sad/Hopeless: No"
})

# ------------------------------------------------------------
# Data check table for SadOrHopeless
# ------------------------------------------------------------
valid_sad_codes = [1, 2]
valid_cig_codes = [1, 2, 3, 4, 5, 6, 7]

sad_valid = sad_raw.isin(valid_sad_codes)
current_cig_valid = current_cig_raw.isin(valid_cig_codes)

final_valid_analysis_rows = (sad_valid & current_cig_valid).sum()

sad_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Code 1 count",
        "Code 2 count",
        "Invalid non-missing values",
        "Final valid sample size",
        "Final valid analysis rows"
    ],
    "value": [
        len(raw),
        int(sad_raw.isna().sum()),
        int(sad_raw.notna().sum()),
        int(sad_raw.eq(1).sum()),
        int(sad_raw.eq(2).sum()),
        int((sad_raw.notna() & ~sad_raw.isin(valid_sad_codes)).sum()),
        int(sad_valid.sum()),
        int(final_valid_analysis_rows)
    ]
})

sad_check.to_csv(TAB_DIR / "01_sad_or_hopeless_data_check.csv", index=False)
display(sad_check)

,metric,value
0,Total rows,14041
1,Missing values,196
2,Non-missing values,13845
3,Code 1 count,4153
4,Code 2 count,9692
5,Invalid non-missing values,0
6,Final valid sample size,13845
7,Final valid analysis rows,13174


### 2.2 Response variable: CurrentCigaretteUse

- **Variable name:** `HowOldAreYou`
- **What the variable measures:** age category of the student.
- **Valid codes used:**  
  - `1` = 12 years old or younger  
  - `2` = 13 years old  
  - `3` = 14 years old  
  - `4` = 15 years old  
  - `5` = 16 years old  
  - `6` = 17 years old  
  - `7` = 18 years old or older
- **Grouping rule used in this project:**  
  - `1, 2, 3` -> `<=14`  
  - `4` -> `15`  
  - `5` -> `16`  
  - `6` -> `17`  
  - `7` -> `18+`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.
- **Final valid sample size for proportion  analysis:** 13323

**Note on age grouping:**  
In the raw data, the numbers of students coded as `1` (12 or younger) and `2` (13 years old) are very small. To make the subgroup comparison more stable and easier to interpret, this project combines codes `1`, `2`, and `3` into a broader group labeled `<=14`. This keeps the younger students in the analysis while reducing instability from very small subgroup counts.

In [3]:
current_cig_raw = raw["CurrentCigaretteUse"]

# ------------------------------------------------------------
# Recode response variable: CurrentCigaretteUse
# 1 = 0 days -> non-smoker
# 2-7 = smoked at least 1 day -> current smoker
# ------------------------------------------------------------
raw["smoker_binary"] = pd.NA
raw.loc[current_cig_raw.eq(1), "smoker_binary"] = 0
raw.loc[current_cig_raw.isin([2, 3, 4, 5, 6, 7]), "smoker_binary"] = 1

raw["current_cigarette_status"] = raw["smoker_binary"].map({
    0: "Non-smoker",
    1: "Current smoker"
})

# ------------------------------------------------------------
# Additional grouped version for exploratory EDA
# Keep Non-smoker and Missing separated
# ------------------------------------------------------------
raw["smoking_freq_group"] = pd.NA

raw.loc[current_cig_raw.eq(1), "smoking_freq_group"] = "Non-smoker (0 days)"
raw.loc[current_cig_raw.isin([2, 3]), "smoking_freq_group"] = "Light (1~5 days)"
raw.loc[current_cig_raw.isin([4, 5]), "smoking_freq_group"] = "Moderate (6~19 days)"
raw.loc[current_cig_raw.isin([6, 7]), "smoking_freq_group"] = "Frequent (20~30 days)"

# ------------------------------------------------------------
# Raw code count table
# ------------------------------------------------------------
current_cig_label_map = {
    1: "0 days",
    2: "1 or 2 days",
    3: "3 to 5 days",
    4: "6 to 9 days",
    5: "10 to 19 days",
    6: "20 to 29 days",
    7: "all 30 days"
}

valid_codes = list(current_cig_label_map.keys())

current_cig_counts = pd.DataFrame({
    "code": valid_codes + ["Missing"],
    "count": [
        int((current_cig_raw == code).sum())
        for code in valid_codes
    ] + [
        int((current_cig_raw.isna() | ~current_cig_raw.isin(valid_codes)).sum())
    ],
    "meaning": list(current_cig_label_map.values()) + ["Missing or invalid"]
})

current_cig_counts.to_csv(TAB_DIR / "01_current_cigarette_use_code_counts.csv", index=False)
display(current_cig_counts)

# ------------------------------------------------------------
# Smoking frequency group count table
# Non-smoker and Missing are separated here
# ------------------------------------------------------------
smoking_freq_display = raw["smoking_freq_group"].fillna("Missing")

freq_order = [
    "Non-smoker (0 days)",
    "Light (1~5 days)",
    "Moderate (6~19 days)",
    "Frequent (20~30 days)",
    "Missing"
]

smoking_freq_group_counts = (
    smoking_freq_display
    .value_counts()
    .reindex(freq_order, fill_value=0)
    .rename_axis("smoking_freq_group")
    .reset_index(name="count")
)

smoking_freq_group_counts.to_csv(TAB_DIR / "01_smoking_freq_group_counts.csv", index=False)
display(smoking_freq_group_counts)

,code,count,meaning
0,1,10734,0 days
1,2,753,1 or 2 days
2,3,375,3 to 5 days
3,4,250,6 to 9 days
4,5,295,10 to 19 days
5,6,229,20 to 29 days
6,7,687,all 30 days
7,Missing,718,Missing or invalid


,smoking_freq_group,count
0,Non-smoker (0 days),10734
1,Light (1~5 days),1128
2,Moderate (6~19 days),545
3,Frequent (20~30 days),916
4,Missing,718


## 3. Rows valid for both SadOrHopeless and CurrentCigaretteUse

In [4]:
analysis = raw.dropna(subset=["sad_binary", "smoker_binary"]).copy()

analysis["sad_binary"] = analysis["sad_binary"].astype(int)
analysis["smoker_binary"] = analysis["smoker_binary"].astype(int)

analysis.to_csv(PROCESSED_PATH, index=False)

print("Rows valid for both SadOrHopeless and CurrentCigaretteUse:", len(analysis))

Rows valid for both SadOrHopeless and CurrentCigaretteUse: 13174
